# Cryptocurrency Hedge & Safe-Haven Analysis
Reproducible companion notebook to the dissertation. Run from the repository root.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src").exists(): root = root.parent
sys.path.insert(0, str(root / "src"))
from analysis import load_prices, make_returns, fit_models
import pandas as pd

## 1. Load Bloomberg price exports and align to SPX trading dates

In [ ]:
prices = load_prices()
returns = make_returns(prices)
returns.count()

## 2. Common-sample correlations

In [ ]:
returns[["SPX", "BTC", "ETH"]].dropna().corr()

## 3. Baseline and extreme-market regressions

In [ ]:
threshold = returns["SPX"].dropna().quantile(.05)
for ticker in ("BTC", "ETH"):
    sample, hedge, haven, _ = fit_models(returns, ticker, threshold)
    print(ticker, "observations:", len(sample), "stress threshold:", threshold)
    print("Hedge coefficients:\n", hedge.params)
    print("Safe haven coefficients:\n", haven.params)
    print("Stress-period beta:", haven.params["SPX"] + haven.params["SPX_x_stress"])

## 4. Rolling correlations

In [ ]:
import matplotlib.pyplot as plt
for ticker in ("BTC", "ETH"):
    sample = returns[["SPX", ticker]].dropna()
    sample["SPX"].rolling(60).corr(sample[ticker]).plot(figsize=(12,3), title=f"{ticker}: 60-day rolling correlation")
    plt.axhline(0, color="grey")
    plt.show()